In [0]:
df = spark.read.csv(
    "/Volumes/workspace/default/superstore-volume/Sample - Superstore.csv",
    header=True,
    inferSchema=True
)
df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [0]:
# Check schema - shows column names and data types
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
# Confirm total number of rows
print(f"Total rows: {df.count()}")

Total rows: 9994


In [0]:
# ─────────────────────────────────────────
# Q3: Remove Duplicate Rows Based on Specific Columns
# ─────────────────────────────────────────

df_dedup = df.dropDuplicates(['Order ID', 'Order Date'])

print(f"Original count : {df.count()}")
print(f"After dedup    : {df_dedup.count()}")

df_dedup.show(5)

Original count : 9994
After dedup    : 5009
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|         State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|    68|CA-2014-106376|2014-12-05|2014-12-10|Standard Class|   BS-11590|    Brendan Sweed|  Corporate|United States|      Gilbert|       Arizona|      8523

In [0]:
# Save Q3 result to query_results folder
df_dedup.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q3_dedup_result"
)
print("Q3 result saved successfully!")

Q3 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q4: Filter by Region = West, GroupBy Category, Avg Sale Amount
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Fix Sales column using try_cast to handle messy values gracefully
df_clean = df.withColumn('Sales',
    F.expr("try_cast(trim(Sales) as double)")
)

# Check how many nulls were created after casting
null_count = df_clean.filter(F.col('Sales').isNull()).count()
print(f"Rows with invalid Sales values: {null_count}")

df_q4 = (df_clean
    .filter(F.col('Region') == 'West')
    .groupBy('Category')
    .agg(F.avg('Sales').alias('avg_sale_amount'))
    .orderBy('avg_sale_amount', ascending=False)
)

df_q4.show()

# Save result
df_q4.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q4_filter_agg_result"
)
print("Q4 result saved successfully!")

Rows with invalid Sales values: 300
+---------------+------------------+
|       Category|   avg_sale_amount|
+---------------+------------------+
|     Technology|422.64417449664444|
|      Furniture| 360.5954042089983|
|Office Supplies|117.48907552370456|
+---------------+------------------+

Q4 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q5: .na.drop() vs .na.fill() for Null Handling
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# First fix Sales column
df_clean = df.withColumn('Sales',
    F.expr("try_cast(trim(Sales) as double)")
)

# .na.drop() - removes entire rows where Sales is null
df_dropped = df_clean.na.drop(subset=['Sales'])
print(f"Original count        : {df_clean.count()}")
print(f"After na.drop()       : {df_dropped.count()}")
print(f"Rows removed          : {df_clean.count() - df_dropped.count()}")

# .na.fill() - fills null Sales values with 0 instead of removing rows
df_filled = df_clean.na.fill({'Sales': 0.0})
print(f"After na.fill()       : {df_filled.count()}")
print(f"Rows kept with fill   : {df_filled.count()}")

df_filled.show(5)

# Save result
df_filled.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q5_null_handling"
)
print("Q5 result saved successfully!")

Original count        : 9994
After na.drop()       : 9694
Rows removed          : 300
After na.fill()       : 9994
Rows kept with fill   : 9994
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520| 

In [0]:
# ─────────────────────────────────────────
# Q6: Count Records per City - Only Cities with Count > 100
# ─────────────────────────────────────────
from pyspark.sql import functions as F

df_q6 = (df
    .groupBy('City')
    .agg(F.count('*').alias('record_count'))
    .filter(F.col('record_count') > 100)
    .orderBy('record_count', ascending=False)
)

df_q6.show()

# Save result
df_q6.coalesce(1).write.mode("overwrite").option("header", "true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q6_city_count"
)
print("Q6 result saved successfully!")

+-------------+------------+
|         City|record_count|
+-------------+------------+
|New York City|         915|
|  Los Angeles|         747|
| Philadelphia|         537|
|San Francisco|         510|
|      Seattle|         428|
|      Houston|         377|
|      Chicago|         314|
|     Columbus|         222|
|    San Diego|         170|
|  Springfield|         163|
|       Dallas|         157|
| Jacksonville|         125|
|      Detroit|         115|
+-------------+------------+

Q6 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q7: Immutability of Spark DataFrames
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Original DataFrame - never gets modified
print(f"Original df columns : {len(df.columns)}")
print(f"Original df rows    : {df.count()}")

# Each transformation returns a NEW DataFrame
df_step1 = df.drop('Row ID')                                      # Drop column
df_step2 = df_step1.withColumnRenamed('Order ID', 'order_id')    # Rename column
df_step3 = df_step2.dropDuplicates(['order_id'])                  # Remove duplicates
df_step4 = df_step3.filter(F.col('Region') == 'West')            # Filter rows

# Prove original is unchanged
print(f"\nAfter all transformations:")
print(f"Original df columns still : {len(df.columns)}")
print(f"Original df rows still    : {df.count()}")

# Show the new cleaned DataFrame
print(f"\nNew cleaned df columns : {len(df_step4.columns)}")
print(f"New cleaned df rows    : {df_step4.count()}")

df_step4.show(5)

# Save result
df_step4.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q7_immutability"
)
print("Q7 result saved successfully!")

Original df columns : 21
Original df rows    : 9994

After all transformations:
Original df columns still : 21
Original df rows still    : 9994

New cleaned df columns : 20
New cleaned df rows    : 1611
+--------------+----------+----------+--------------+-----------+----------------+--------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|      order_id|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name| Segment|      Country|       City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount|  Profit|
+--------------+----------+----------+--------------+-----------+----------------+--------+-------------+-----------+----------+-----------+------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|CA-2016-154508|2016-11-16|2016-11-20|Standard Class|

In [0]:
# ─────────────────────────────────────────
# Q8: Filter Age Between 18-30 AND Subscription = Premium
# Note: Creating sample data to demonstrate this concept
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Create sample DataFrame with age and subscription columns
sample_data = [
    ("Alice", 22, "Premium"),
    ("Bob", 35, "Basic"),
    ("Charlie", 28, "Premium"),
    ("Diana", 17, "Premium"),
    ("Eve", 30, "Basic"),
    ("Frank", 25, "Premium"),
    ("Grace", 45, "Premium"),
    ("Henry", 19, "Basic"),
    ("Isla", 27, "Premium"),
    ("Jack", 30, "Premium"),
]

columns = ["name", "age", "subscription"]
df_sample = spark.createDataFrame(sample_data, columns)

print("Sample DataFrame:")
df_sample.show()

# Filter age between 18-30 AND subscription = Premium
df_q8 = df_sample.filter(
    (F.col('age').between(18, 30)) &
    (F.col('subscription') == 'Premium')
)

print("Filtered Result - Age 18-30 AND Premium subscribers:")
df_q8.show()

# Save result
df_q8.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q8_filter_result"
)
print("Q8 result saved successfully!")

Sample DataFrame:
+-------+---+------------+
|   name|age|subscription|
+-------+---+------------+
|  Alice| 22|     Premium|
|    Bob| 35|       Basic|
|Charlie| 28|     Premium|
|  Diana| 17|     Premium|
|    Eve| 30|       Basic|
|  Frank| 25|     Premium|
|  Grace| 45|     Premium|
|  Henry| 19|       Basic|
|   Isla| 27|     Premium|
|   Jack| 30|     Premium|
+-------+---+------------+

Filtered Result - Age 18-30 AND Premium subscribers:
+-------+---+------------+
|   name|age|subscription|
+-------+---+------------+
|  Alice| 22|     Premium|
|Charlie| 28|     Premium|
|  Frank| 25|     Premium|
|   Isla| 27|     Premium|
|   Jack| 30|     Premium|
+-------+---+------------+

Q8 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q9: Why Handle Nulls Before Aggregations
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Fix Sales column
df_clean = df.withColumn('Sales',
    F.expr("try_cast(trim(Sales) as double)")
)

# WITHOUT null handling - avg() silently ignores nulls
avg_with_nulls = df_clean.agg(
    F.avg('Sales').alias('avg_ignoring_nulls'),
    F.count('Sales').alias('count_non_null')
).collect()[0]

print(f"Avg Sales (nulls ignored) : {round(avg_with_nulls['avg_ignoring_nulls'], 2)}")
print(f"Rows used in calculation  : {avg_with_nulls['count_non_null']} out of {df_clean.count()}")

# WITH null handling - fill nulls with 0 first
df_filled = df_clean.na.fill({'Sales': 0.0})
avg_after_fill = df_filled.agg(
    F.avg('Sales').alias('avg_after_fill'),
    F.count('Sales').alias('count_total')
).collect()[0]

print(f"\nAvg Sales (nulls as 0)    : {round(avg_after_fill['avg_after_fill'], 2)}")
print(f"Rows used in calculation  : {avg_after_fill['count_total']} out of {df_clean.count()}")

print(f"\nDifference in avg         : {round(avg_with_nulls['avg_ignoring_nulls'] - avg_after_fill['avg_after_fill'], 2)}")

# Save result
df_filled.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q9_null_aggregation"
)
print("Q9 result saved successfully!")

Avg Sales (nulls ignored) : 234.42
Rows used in calculation  : 9694 out of 9994

Avg Sales (nulls as 0)    : 227.38
Rows used in calculation  : 9994 out of 9994

Difference in avg         : 7.04
Q9 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q10: Cast and Rename Column - raw_timestamp to event_time
# ─────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType

# Cast Ship Date to TimestampType and rename to event_time
df_q10 = (df
    .withColumn('event_time',
        F.col('Ship Date').cast(TimestampType())
    )
    .drop('Ship Date')
)

# Verify the change
print("New schema for relevant columns:")
df_q10.select('Order Date', 'event_time').printSchema()

df_q10.select('Order Date', 'event_time').show(5)

# Save result
df_q10.select('Order Date', 'event_time').coalesce(1)\
    .write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q10_cast_rename"
)
print("Q10 result saved successfully!")

New schema for relevant columns:
root
 |-- Order Date: date (nullable = true)
 |-- event_time: timestamp (nullable = true)

+----------+-------------------+
|Order Date|         event_time|
+----------+-------------------+
|2016-11-08|2016-11-11 00:00:00|
|2016-11-08|2016-11-11 00:00:00|
|2016-06-12|2016-06-16 00:00:00|
|2015-10-11|2015-10-18 00:00:00|
|2015-10-11|2015-10-18 00:00:00|
+----------+-------------------+
only showing top 5 rows
Q10 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q11: Shuffle Process in groupBy - Wide Transformation
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# This groupBy triggers a SHUFFLE - wide transformation
df_q11 = df.groupBy('Region').agg(
    F.count('*').alias('total_records'),
    F.countDistinct('Customer ID').alias('unique_customers')
)

# Show the result
df_q11.show()

# Show execution plan - proves shuffle happened
print("\nExecution Plan (look for Exchange = Shuffle):")
df_q11.explain()

# Save result
df_q11.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q11_shuffle_demo"
)
print("Q11 result saved successfully!")

+-------+-------------+----------------+
| Region|total_records|unique_customers|
+-------+-------------+----------------+
|   East|         2848|             674|
|  South|         1620|             512|
|Central|         2323|             629|
|   West|         3203|             686|
+-------+-------------+----------------+


Execution Plan (look for Exchange = Shuffle):
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonGroupingAgg(keys=[Region#13551], functions=[finalmerge_count(merge count#13593L) AS count(1)#13590L, finalmerge_count(distinct merge count#13595L) AS count(Customer ID)#13591L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#11289]
               +- PhotonShuffleExchangeSink hashpartitioning(Region#13551, 16)
                  +- PhotonGroupingAgg(keys=[Region#13551], functions=[merge_count(merge count#13593L) AS count#13593

In [0]:
# ─────────────────────────────────────────
# Q12: Remove Rows where Email is Null OR Username is Empty
# Note: Creating sample data to demonstrate this concept
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Create sample DataFrame
sample_data = [
    ("alice123", "alice@gmail.com"),
    ("bob456",   None),
    ("",         "charlie@gmail.com"),
    ("diana789", "diana@gmail.com"),
    (None,       "eve@gmail.com"),
    ("",         None),
    ("frank101", "frank@gmail.com"),
]

columns = ["username", "email"]
df_q12 = spark.createDataFrame(sample_data, columns)

print("Original Sample DataFrame:")
df_q12.show()

# Remove rows where email is null OR username is empty
df_cleaned = df_q12.filter(
    F.col('email').isNotNull() &
    (F.trim(F.col('username')) != '')
)

print("After removing null emails and empty usernames:")
df_cleaned.show()

print(f"Original rows : {df_q12.count()}")
print(f"Cleaned rows  : {df_cleaned.count()}")
print(f"Rows removed  : {df_q12.count() - df_cleaned.count()}")

# Save result
df_cleaned.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q12_null_empty_filter"
)
print("Q12 result saved successfully!")


Original Sample DataFrame:
+--------+-----------------+
|username|            email|
+--------+-----------------+
|alice123|  alice@gmail.com|
|  bob456|             NULL|
|        |charlie@gmail.com|
|diana789|  diana@gmail.com|
|    NULL|    eve@gmail.com|
|        |             NULL|
|frank101|  frank@gmail.com|
+--------+-----------------+

After removing null emails and empty usernames:
+--------+---------------+
|username|          email|
+--------+---------------+
|alice123|alice@gmail.com|
|diana789|diana@gmail.com|
|frank101|frank@gmail.com|
+--------+---------------+

Original rows : 7
Cleaned rows  : 3
Rows removed  : 4
Q12 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q13: Using .agg() for Multiple Statistics at Once
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# Fix Sales column first
df_clean = df.withColumn('Sales',
    F.expr("try_cast(trim(Sales) as double)")
)

# Calculate multiple stats in ONE .agg() call
price_stats = df_clean.agg(
    F.min('Sales').alias('min_sales'),
    F.max('Sales').alias('max_sales'),
    F.avg('Sales').alias('mean_sales'),
    F.stddev('Sales').alias('std_sales'),
    F.count('Sales').alias('count_non_null')
)

print("Overall Sales Statistics:")
price_stats.show()

# Per category stats using groupBy + agg together
category_stats = df_clean.groupBy('Category').agg(
    F.min('Sales').alias('min_sales'),
    F.max('Sales').alias('max_sales'),
    F.avg('Sales').alias('mean_sales'),
    F.sum('Sales').alias('total_sales')
).orderBy('total_sales', ascending=False)

print("Sales Statistics per Category:")
category_stats.show()

# Save result
category_stats.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q13_agg_stats"
)
print("Q13 result saved successfully!")

Overall Sales Statistics:
+---------+---------+------------------+-----------------+--------------+
|min_sales|max_sales|        mean_sales|        std_sales|count_non_null|
+---------+---------+------------------+-----------------+--------------+
|    0.444| 22638.48|234.41818199917438|631.7890112674339|          9694|
+---------+---------+------------------+-----------------+--------------+

Sales Statistics per Category:
+---------------+---------+---------+------------------+-----------------+
|       Category|min_sales|max_sales|        mean_sales|      total_sales|
+---------------+---------+---------+------------------+-----------------+
|     Technology|     0.99| 22638.48| 454.5405475802061|835900.0669999991|
|      Furniture|    1.892| 4416.174|353.44593119575745|733046.8613000009|
|Office Supplies|    0.444|  9892.74|121.69225531914908|703502.9280000009|
+---------------+---------+---------+------------------+-----------------+

Q13 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q14: Risk of inferSchema=True with Inconsistent Date Formats
# ─────────────────────────────────────────
from pyspark.sql import functions as F

sample_data = [
    ("2024-01-15",  "Alice",   250.0),
    ("15/01/2024",  "Bob",     180.0),
    ("Jan 15 2024", "Charlie", 320.0),
    ("2024-01-20",  "Diana",   410.0),
    ("20-01-2024",  "Eve",     150.0),
]

columns = ["order_date", "customer", "amount"]
df_messy = spark.createDataFrame(sample_data, columns)

# WRONG approach - single format, mismatched dates become NULL
df_wrong = df_messy.withColumn('parsed_date_wrong',
    F.expr("try_to_date(order_date, 'yyyy-MM-dd')")
)

print("Wrong approach - NULLs appear for non-matching formats:")
df_wrong.select('order_date', 'parsed_date_wrong').show()

null_dates = df_wrong.filter(F.col('parsed_date_wrong').isNull()).count()
print(f"Silently lost rows: {null_dates} out of {df_messy.count()}")

# CORRECT approach - try multiple formats using coalesce
df_correct = df_messy.withColumn('parsed_date_correct',
    F.coalesce(
        F.expr("try_to_date(order_date, 'yyyy-MM-dd')"),
        F.expr("try_to_date(order_date, 'dd/MM/yyyy')"),
        F.expr("try_to_date(order_date, 'MMM dd yyyy')"),
        F.expr("try_to_date(order_date, 'dd-MM-yyyy')")
    )
)

print("Correct approach - all dates parsed successfully:")
df_correct.select('order_date', 'parsed_date_correct').show()

null_dates_correct = df_correct.filter(
    F.col('parsed_date_correct').isNull()
).count()
print(f"Lost rows with correct approach: {null_dates_correct} out of {df_messy.count()}")

# Save result
df_correct.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q14_messy_dates"
)
print("Q14 result saved successfully!")

Wrong approach - NULLs appear for non-matching formats:
+-----------+-----------------+
| order_date|parsed_date_wrong|
+-----------+-----------------+
| 2024-01-15|       2024-01-15|
| 15/01/2024|             NULL|
|Jan 15 2024|             NULL|
| 2024-01-20|       2024-01-20|
| 20-01-2024|             NULL|
+-----------+-----------------+

Silently lost rows: 3 out of 5
Correct approach - all dates parsed successfully:
+-----------+-------------------+
| order_date|parsed_date_correct|
+-----------+-------------------+
| 2024-01-15|         2024-01-15|
| 15/01/2024|         2024-01-15|
|Jan 15 2024|         2024-01-15|
| 2024-01-20|         2024-01-20|
| 20-01-2024|         2024-01-20|
+-----------+-------------------+

Lost rows with correct approach: 0 out of 5
Q14 result saved successfully!


In [0]:
# ─────────────────────────────────────────
# Q15: Final End-to-End Processing Pipeline
# Step 1 - Remove duplicates
# Step 2 - Fill null Sales with 0
# Step 3 - Group by Category to calculate total revenue
# ─────────────────────────────────────────
from pyspark.sql import functions as F

# STEP 1 - Remove duplicates
df_step1 = df.dropDuplicates(['Order ID', 'Order Date'])
print(f"Step 1 - After dedup        : {df_step1.count()} rows")

# STEP 2 - Fix Sales AND Quantity columns, fill nulls with 0
df_step2 = (df_step1
    .withColumn('Sales',
        F.expr("try_cast(trim(Sales) as double)"))
    .withColumn('Quantity',
        F.expr("try_cast(trim(Quantity) as int)"))
    .na.fill({'Sales': 0.0, 'Quantity': 0})
)
print(f"Step 2 - After null fill    : {df_step2.count()} rows")

# Check how many invalid Quantity values were found
invalid_qty = df_step1.withColumn('Quantity',
    F.expr("try_cast(trim(Quantity) as int)")
).filter(F.col('Quantity').isNull()).count()
print(f"Step 2 - Invalid Quantity values fixed: {invalid_qty}")

# STEP 3 - Add revenue column and group by Category
df_step3 = df_step2.withColumn('Revenue',
    F.col('Sales') * F.col('Quantity')
)

final_result = (df_step3
    .groupBy('Category')
    .agg(
        F.sum('Revenue').alias('total_revenue'),
        F.count('*').alias('total_orders'),
        F.avg('Sales').alias('avg_sale_amount'),
        F.sum('Sales').alias('total_sales')
    )
    .orderBy('total_revenue', ascending=False)
)

print("\nFinal Pipeline Result:")
final_result.show()

# Save final result
final_result.coalesce(1).write.mode("overwrite").option("header","true").csv(
    "/Volumes/workspace/default/superstore-volume/query_results/q15_final_pipeline"
)
print("Q15 Final Pipeline result saved successfully!")

Step 1 - After dedup        : 5009 rows
Step 2 - After null fill    : 5009 rows
Step 2 - Invalid Quantity values fixed: 149

Final Pipeline Result:
+---------------+------------------+------------+------------------+-----------------+
|       Category|     total_revenue|total_orders|   avg_sale_amount|      total_sales|
+---------------+------------------+------------+------------------+-----------------+
|      Furniture|1957951.9122999997|        1102| 335.3057991833038|369506.9907000008|
|     Technology| 1823107.056000002|         864|440.50479861111086|380596.1459999998|
|Office Supplies|1701981.8560000057|        3043|110.97770949720673|337705.1700000001|
+---------------+------------------+------------+------------------+-----------------+

Q15 Final Pipeline result saved successfully!
